# Faruq-v3 — GEO-SHARED60 vs GEO-FAM35×3 Exact-Capacity Validation

Exploratory architecture validation on seeds 42/123/2026. Both candidates add exactly **849 trainable parameters** and use the same detached predicted-box geometry. This is not independent confirmation because the family hypothesis was derived from these development validation results. **No locked test.**

Execution follows the latest notebook style: fresh clone, force-remount Drive, focused tests, static preflight, compact 60-second progress, full log file, and failure-tail printing. Frozen criteria are committed in `docs/FARUQ_V3_GEOMETRY_FAMILY_FACTORIZATION_PROTOCOL.md`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/circle-cpe-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-geometry-conditioning-paired-confirmation-v1/val_reports/geometry_conditioning_paired_three_seed_confirmation.json',
    'experiments/faruq-v3-geometry-family-effect-decomposition-v1/geometry_family_effect_decomposition.json',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/experiment_manifest.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/experiment_manifest.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/val_reports/D0FT_seed123_val.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/experiment_manifest.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/val_reports/D0FT_seed2026_val.json',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
CONFIRMATION = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
DECOMPOSITION = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
for rel in REQUIRED[3:]: require_project_artifact(PROJECT_ROOT, rel)
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT/'faruq_grouped_summary.json').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT/'faruq_grouped_summary.json').is_file()
assert not (DATA_ROOT/'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT/'experiments/faruq-v3-geometry-family-factorization-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU      :', torch.cuda.get_device_name(0))
print('CONFIRM  :', CONFIRMATION)
print('DECOMP   :', DECOMPOSITION)
print('OUTPUT   :', OUTPUT_ROOT)


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_geometry_family_factorization.py']
print('FOCUSED TEST:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
base = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_geometry_family_factorization',
    '--data-root', str(DATA_ROOT),
    '--project-root', str(PROJECT_ROOT),
    '--confirmation-summary', str(CONFIRMATION),
    '--family-decomposition', str(DECOMPOSITION),
    '--output-root', str(OUTPUT_ROOT),
    '--device', '0',
]
static_cmd = base + ['--stage','static']
print('STATIC PREFLIGHT:', ' '.join(static_cmd), flush=True)
subprocess.run(static_cmd, cwd=REPO, check=True)
static = json.loads((OUTPUT_ROOT/'static_preflight.json').read_text(encoding='utf-8'))
print(json.dumps(static, indent=2, ensure_ascii=False))
assert static['decision'] == 'PASS', 'Static family-factorization preflight gagal; training diblokir.'


In [ ]:
import csv

train_cmd = base + ['--stage','train','--authorize-training']
TRAIN_LOG = Path('/content/geometry_family_factorization.log')

def compact_progress():
    rows = []
    for seed in (42, 123, 2026):
        for arm in ('GEO-SHARED60', 'GEO-FAM35x3'):
            report = OUTPUT_ROOT/'val_reports'/f'{arm}_seed{seed}_val.json'
            history = OUTPUT_ROOT/f'{arm}_seed{seed}'/'results.csv'
            if report.is_file():
                rows.append(f'{arm}s{seed}=selesai')
            elif history.is_file():
                with history.open(newline='', encoding='utf-8') as stream:
                    epochs = list(csv.DictReader(stream))
                current = epochs[-1]['epoch'] if epochs else '0'
                rows.append(f'{arm}s{seed}={current}/50')
            else:
                rows.append(f'{arm}s{seed}=menunggu')
    return ', '.join(rows)

print('GEO FAMILY FACTORIZATION dimulai; log lengkap:', TRAIN_LOG, flush=True)
started = time.monotonic()
with TRAIN_LOG.open('w', encoding='utf-8') as log_stream:
    process = subprocess.Popen(train_cmd, cwd=REPO, text=True, stdout=log_stream, stderr=subprocess.STDOUT)
    while process.poll() is None:
        elapsed = (time.monotonic() - started) / 60
        print(f'[GEO-FAMILY {elapsed:.1f} menit] {compact_progress()}', flush=True)
        time.sleep(60)
return_code = process.returncode
if return_code != 0:
    tail = TRAIN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-100:]
    print('\n'.join(tail), flush=True)
    raise RuntimeError(f'GEO family factorization gagal, return code={return_code}; log={TRAIN_LOG}')
print('GEO FAMILY FACTORIZATION selesai:', compact_progress(), flush=True)


In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT/'val_reports/geometry_family_factorization_three_seed.json'
assert SUMMARY.is_file(), SUMMARY
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False and result['test_opened'] is False
rows = []
for seed in result['seeds']:
    record = result['per_seed'][str(seed)]
    for arm in ('GEO-SHARED60','GEO-FAM35x3'):
        values = record['results'][arm]
        rows.append({
            'seed': seed, 'model': arm,
            'Macro': values['macro_map50_95'],
            'Bottom3': values['bottom3_class_map50_95'],
            'Worst': values['worst_class_map50_95'],
            'SizeMean': values['size_class_mean_map50_95'],
        })
display(pd.DataFrame(rows).style.format({'Macro':'{:.2%}','Bottom3':'{:.2%}','Worst':'{:.2%}','SizeMean':'{:.2%}'}))

delta_rows = []
for metric, values in result['aggregate'].items():
    delta_rows.append({'metric': metric, 'mean_delta': values['delta_mean'], 'improved_seeds': values['improved_seeds'], 'min_delta': values['delta_min'], 'max_delta': values['delta_max']})
display(pd.DataFrame(delta_rows).style.format({'mean_delta':'{:+.2%}','min_delta':'{:+.2%}','max_delta':'{:+.2%}'}))

family_rows = []
for family, values in result['family_aggregate'].items():
    family_rows.append({'family': family, 'mean_delta': values['delta_mean'], 'improved_seeds': values['improved_seeds'], 'min_delta': values['delta_min'], 'max_delta': values['delta_max']})
display(pd.DataFrame(family_rows).style.format({'mean_delta':'{:+.2%}','min_delta':'{:+.2%}','max_delta':'{:+.2%}'}))

print('SCIENTIFIC STATUS:', result['scientific_status'])
print('CRITERIA:', json.dumps(result['criteria'], indent=2, ensure_ascii=False))
print('DECISION:', result['decision'])
print('NEXT:', result['next_action'])
print('SUMMARY:', SUMMARY)
print('Jangan membuka locked test dari notebook ini.')
